In [ ]:
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Conv2D, MaxPooling2D, Flatten, Dense, Lambda
import numpy as np
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.losses import BinaryCrossentropy
import matplotlib.pyplot as plt


def build_siamese_network(input_shape):
    # Define Input
    input_layer = Input(shape=input_shape)

    # CNN Feature Extractor
    x = Conv2D(64, (3, 3), activation='relu')(input_layer)
    x = MaxPooling2D(pool_size=(2, 2))(x)
    x = Conv2D(128, (3, 3), activation='relu')(x)
    x = MaxPooling2D(pool_size=(2, 2))(x)
    x = Flatten()(x)
    x = Dense(256, activation='relu')(x)
    return Model(input_layer, x)

# Define input shape
input_shape = (105, 105, 1)  # For grayscale images
base_network = build_siamese_network(input_shape)

from tensorflow.keras.layers import Lambda
import tensorflow.keras.backend as K

# Function to compute Euclidean distance
def euclidean_distance(vectors):
    x, y = vectors
    sum_square = K.sum(K.square(x - y), axis=1, keepdims=True)
    return K.sqrt(K.maximum(sum_square, K.epsilon()))

# Input pairs
input_a = Input(shape=input_shape)
input_b = Input(shape=input_shape)

# Extract features
feature_a = base_network(input_a)
feature_b = base_network(input_b)

# Compute distance
distance = Lambda(euclidean_distance)([feature_a, feature_b])

# Define Siamese Model
siamese_model = Model(inputs=[input_a, input_b], outputs=distance)

# Define contrastive loss
def contrastive_loss(y_true, y_pred):
    margin = 1
    return K.mean(y_true * K.square(y_pred) + (1 - y_true) * K.square(K.maximum(margin - y_pred, 0)))

# Compile the model
siamese_model.compile(loss=contrastive_loss, optimizer=Adam(learning_rate=0.0001))

# Train the model (Assuming X_train1, X_train2, Y_train are preprocessed datasets)
# siamese_model.fit([X_train1, X_train2], Y_train, epochs=10, batch_size=32)ß

In [ ]:
# prompt: save my siamese_model

siamese_model.save('siamese_model.h5')


In [ ]:
import cv2
import numpy as np

def preprocess_image(image_path, target_size=(105, 105)):
    img = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)  # Load as grayscale
    img = cv2.resize(img, target_size)  # Resize
    img = img.astype("float32") / 255.0  # Normalize
    img = np.expand_dims(img, axis=-1)  # Add channel dimension
    return img


In [ ]:
image1 = preprocess_image("/content/sign.jpg")
image2 = preprocess_image("/content/7kb.jpg")

# Expand dimensions to match model input shape (batch size 1)
X_test1 = np.expand_dims(image1, axis=0)
X_test2 = np.expand_dims(image2, axis=0)


In [ ]:

import tensorflow as tf
print(tf.__version__) # Print the TensorFlow version


import os
os.environ['TF_FORCE_GPU_ALLOW_GROWTH'] = 'true'  # Allow GPU memory growth

# Check if GPU is available
print("Num GPUs Available: ", len(tf.config.list_physical_devices('GPU')))

# Preprocess the images
# ... (Your existing preprocess_image function and image loading code) ...


# Try predicting again
distance = siamese_model.predict([X_test1, X_test2])
print(f"Distance between images: {distance[0][0]}")

threshold = 0.1  # Adjust based on your dataset
if distance[0][0] < threshold:
    print("Images are similar")
else:
    print("Images are different")

2.18.0
Num GPUs Available:  1
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 33ms/step
Distance between images: 0.10424400120973587
Images are different
